# 🔍 01 — RAG Pipeline: İstanbul Bilgi Erişim Sistemi

Bu notebook İstanbul belgelerini vektöre dönüştürür ve FAISS indeksi oluşturur.

## 📂 Adım 1: Dataset Yükle

In [3]:
from datasets import load_from_disk

dataset = load_from_disk("rag_data/istanbul_dataset")
print(f"✅ {len(dataset)} İstanbul kategorisi yüklendi")


✅ 23 İstanbul kategorisi yüklendi


## ✂️ Adım 2: Metinleri Chunk'lara Böl

In [5]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

all_chunks = []
chunk_meta = []


## 🧠 Adım 3: Embedding Oluştur (Sentence Transformers)

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Model yüklendi: paraphrase-multilingual-MiniLM-L12-v2")

print("🔄 Embedding oluşturuluyor...")
embeddings = model.encode(all_chunks, batch_size=32, show_progress_bar=True)
embeddings = embeddings.astype("float32")
print(f"✅ Embedding boyutu: {embeddings.shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2420.11it/s]


✅ Model yüklendi: paraphrase-multilingual-MiniLM-L12-v2
🔄 Embedding oluşturuluyor...


Batches: 0it [00:00, ?it/s]

✅ Embedding boyutu: (0,)


## 📦 Adım 4: FAISS İndeksi Oluştur ve Kaydet

In [7]:
import faiss
import pickle

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"✅ FAISS indeksi oluşturuldu: {index.ntotal} vektör")

faiss.write_index(index, "rag_data/istanbul_index.faiss")
with open("rag_data/chunks.pkl", "wb") as f:
    pickle.dump({"chunks": all_chunks, "meta": chunk_meta}, f)

print("✅ Kaydedildi: rag_data/istanbul_index.faiss")
print("✅ Kaydedildi: rag_data/chunks.pkl")

IndexError: tuple index out of range

## 🧪 Adım 5: Arama Testi

In [8]:
def ara(soru, k=3):
    q_vec = model.encode([soru]).astype("float32")
    distances, indices = index.search(q_vec, k)
    print(f"🔍 Soru: {soru}")
    print("-" * 60)
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        print(f"[{rank+1}] Kaynak: {chunk_meta[idx]['kaynak']} (mesafe: {dist:.2f})")
        print(f"{all_chunks[idx][:200]}...")

ara("Ayasofya kaç yılında yapıldı?")
ara("İstanbul'dan Asya yakasına nasıl geçerim?")

NameError: name 'index' is not defined